# Train the Edmund wake word model

Replaces the Pi's wake gate, which decodes a general ASR model (Vosk) against a grammar whose
entire vocabulary is the wake words plus `[unk]`, and fires on its *partial* hypotheses.
Ambient speech has almost nothing to decode onto, so it lands on "edmund" — which is why
kitchen conversation and the Alexa in the same room kept opening turns. There is no
confidence value anywhere in that design, so there is nothing to threshold.

This produces an openWakeWord model instead: trained on the phrase, with negative training
against ~2000 hours of speech/noise/music, emitting a **calibrated 0–1 confidence per 80 ms
frame** that can be tuned against a false-accepts-per-hour target.

**Before running:** Runtime → Change runtime type → **GPU**.

**Runtime:** ~1.5–3 h. Downloads ~20 GB (Colab has the disk; the Mac does not, which is why
this runs here).

**Output:** `edmund.onnx`, downloaded at the end.

### Deviations from openWakeWord's own notebook, and why

Its upstream notebook no longer runs as written. Each of these was verified against the
current source or the live URL, not assumed:

- `piper-phonemize` → **`espeak-phonemizer`**. The dscripka fork's `requirements.txt` asks for
  espeak-phonemizer and `generate_samples.py` imports it; piper-phonemize publishes no wheel
  for Colab's Python 3.12 and is never imported.
- `webrtcvad` → **`webrtcvad-wheels`**. Same module, but sdist-only upstream and it does not
  compile on 3.12.
- **TTS model is `en-us-libritts-high.pt` (v1.0.0)**, not v2.0.0's `libritts_r-medium`.
  `train.py` calls `generate_samples()` with no `model=` argument, so only the default
  filename is ever looked up.
- **`tensorflow-cpu`, `tensorflow_probability`, `onnx_tf` removed.** They are imported inside
  `convert_onnx_to_tflite()`, which runs only under `--convert_to_tflite`. We export ONNX,
  which is what the Pi runs. `deep-phonemizer` removed too — not imported anywhere.
- **AudioSet `.tar` files no longer exist** (the repo is parquet now); the old `wget` 404s.
- **FMA source swapped** from `rudraml/fma` to `benjamin-paine/free-music-archive-small`.
  The former is a loading-script dataset, which needs `trust_remote_code` on datasets 3.x and
  was removed outright in 4.x.
- **Room impulse responses skip `datasets` entirely** — that repo is plain 16 kHz wavs, so
  they are pulled directly. This is what was crashing with `AudioDecoder is not subscriptable`.
- **`datasets` pinned `<4.0`** for the dict-style audio API. Installed last so nothing bumps it.

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out.split("\n")[8] if out else
      "NO GPU — Runtime > Change runtime type > GPU, then Run all again.")

## 1. Environment

In [ ]:
# espeak-phonemizer bundles its own libespeak-ng.so, but the system package is cheap insurance.
!apt-get -qq update && apt-get -qq install -y espeak-ng libespeak-ng-dev

![ -d piper-sample-generator ] || git clone -q https://github.com/dscripka/piper-sample-generator
# train.py calls generate_samples() with no model= argument, so only the generator's default
# path is ever read: models/en-us-libritts-high.pt (rhasspy v1.0.0, ~243 MB).
!wget -q --show-progress -O piper-sample-generator/models/en-us-libritts-high.pt \
  'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'

# webrtcvad-wheels, not webrtcvad: same module name, but upstream is sdist-only and will not
# compile on Python 3.12.
!pip install -q espeak-phonemizer webrtcvad-wheels

![ -d openwakeword ] || git clone -q https://github.com/dscripka/openwakeword
!pip install -q -e ./openwakeword
!pip install -q mutagen torchinfo torchmetrics speechbrain audiomentations \
  torch-audiomentations acoustics pronouncing librosa

# Last, so the editable install above cannot pull a 4.x/5.x back over it. 4.0 replaced the
# audio dict with a torchcodec AudioDecoder, which every loop below would have to special-case.
!pip install -q "datasets<4.0"

import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q -P ./openwakeword/openwakeword/resources/models \
  https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx \
  https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite \
  https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx \
  https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite

In [ ]:
import os, sys, uuid, yaml, scipy, scipy.io.wavfile, datasets, numpy as np, torch
from pathlib import Path
from tqdm import tqdm
from huggingface_hub import snapshot_download

In [ ]:
# Fail fast. Everything below is needed later, and discovering a missing import two hours in --
# after a 16.5 GB download and the clip generation -- is the expensive way to find out.
import importlib
sys.path.insert(0, os.path.abspath("./piper-sample-generator"))

problems = []
for m in ["torch", "torchinfo", "torchmetrics", "speechbrain", "audiomentations",
          "torch_audiomentations", "acoustics", "mutagen", "pronouncing", "datasets",
          "webrtcvad", "espeak_phonemizer", "librosa", "openwakeword"]:
    try:
        importlib.import_module(m)
    except Exception as e:
        problems.append(f"  import {m}: {type(e).__name__}: {e}")

try:
    from generate_samples import generate_samples
except Exception as e:
    problems.append(f"  generate_samples: {type(e).__name__}: {e}")

if int(datasets.__version__.split(".")[0]) >= 4:
    problems.append(f"  datasets is {datasets.__version__}; needs <4.0 (restart the session)")

mdl = "piper-sample-generator/models/en-us-libritts-high.pt"
if not os.path.exists(mdl):
    problems.append(f"  TTS model missing: {mdl}")
elif os.path.getsize(mdl) < 200_000_000:
    problems.append(f"  TTS model truncated: {os.path.getsize(mdl)} bytes, expected ~243 MB")

print("\n".join(["PROBLEMS:"] + problems) if problems else
      f"OK — python {sys.version.split()[0]}, torch {torch.__version__}, "
      f"datasets {datasets.__version__}, TTS model {os.path.getsize(mdl)//10**6} MB")

## 2. Data

Room impulse responses (to make synthetic speech sound like a room), background noise and
music (to mix in), ~2000 h of precomputed negative features, and a validation set for
estimating the false-positive rate.

In [ ]:
# Room impulse responses — what makes a clean TTS clip sound like it crossed a kitchen.
# This repo is plain 16 kHz wavs, so it is pulled directly; routing it through `datasets`
# bought nothing and is what raised "AudioDecoder is not subscriptable".
rir_repo = snapshot_download(repo_id="davidscripka/MIT_environmental_impulse_responses",
                             repo_type="dataset")
RIR_DIR = os.path.join(rir_repo, "16khz")
print("room impulse responses:", len(list(Path(RIR_DIR).glob("*.wav"))), "->", RIR_DIR)

In [ ]:
# AudioSet: general environmental noise. The repo is parquet now -- the bal_train*.tar files
# the upstream notebook downloads were removed and that URL 404s.
os.makedirs("./audioset_16k", exist_ok=True)
N_AUDIOSET = 4000

aud = datasets.load_dataset("agkphysics/AudioSet", "balanced", split="train", streaming=True)
aud = aud.cast_column("audio", datasets.Audio(sampling_rate=16000))
secs = 0.0
for i, row in enumerate(tqdm(aud, total=N_AUDIOSET)):
    if i >= N_AUDIOSET:
        break
    a = row["audio"]
    name = Path(a["path"]).stem + ".wav"
    scipy.io.wavfile.write(os.path.join("./audioset_16k", name), 16000,
                           (np.clip(a["array"], -1, 1) * 32767).astype(np.int16))
    secs += len(a["array"]) / 16000
print(f"audioset: {i} clips, {secs/3600:.1f} h")

In [ ]:
# Music, specifically: the mirror lives in a kitchen with a speaker in it, and a model that
# never heard music will fire on it. Parquet-native mirror of FMA -- rudraml/fma is a
# loading-script dataset, which 3.x gates behind trust_remote_code and 4.x removed outright.
os.makedirs("./fma", exist_ok=True)
N_FMA = 500

fma = datasets.load_dataset("benjamin-paine/free-music-archive-small", split="train",
                            streaming=True)
fma = fma.cast_column("audio", datasets.Audio(sampling_rate=16000))
secs = 0.0
for i, row in enumerate(tqdm(fma, total=N_FMA)):
    if i >= N_FMA:
        break
    a = row["audio"]
    scipy.io.wavfile.write(os.path.join("./fma", f"fma_{i:05d}.wav"), 16000,
                           (np.clip(a["array"], -1, 1) * 32767).astype(np.int16))
    secs += len(a["array"]) / 16000
print(f"fma: {i} clips, {secs/3600:.1f} h")

In [ ]:
# Precomputed negative features (~16.5 GB, ~2000 h from ACAV100M) plus the validation set.
# This file is the single reason training does not run on the Mac.
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
!ls -lh *.npy

## 3. Configuration

In [ ]:
config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)

# Both forms are trained so the bare name keeps working. "edmund" alone is two syllables,
# which is short for a wake word and is a real accuracy ceiling regardless of engine --
# openWakeWord's own models are "hey jarvis", "hey mycroft", "alexa". If false accepts
# persist after threshold tuning, prefer "hey edmund", which this model already knows.
config["target_phrase"] = ["edmund", "hey edmund"]
config["model_name"]    = "edmund"

# Hard negatives. The trainer already generates phonetically overlapping words on its own;
# these are the ones actually OBSERVED waking the mirror, plus the device sharing the room.
# Adding to THIS list is the maintenance path when a new false wake shows up -- not
# hand-tuning the threshold, and not extending the string matcher on the Mac.
config["custom_negative_phrases"] = [
    "alexa", "hey alexa", "echo", "amazon",
    "admin", "almond", "amen", "demand", "edmonton", "adman",
    "second", "seven", "a moment", "the moment", "add mint",
    "let me", "and then", "at me", "help me", "woman", "human",
]

# Production scale. The shipped config calls 20,000 a minimum and "often 100,000+ is best";
# n_samples is the highest-leverage knob here. On an A100 you can afford to raise it, and
# to raise tts_batch_size from 50 to ~200.
config["n_samples"]           = 30000
config["n_samples_val"]       = 5000
config["steps"]               = 50000
config["augmentation_rounds"] = 2

config["rir_paths"]        = [RIR_DIR]
config["background_paths"] = ["./audioset_16k", "./fma"]
config["background_paths_duplication_rate"] = [1, 1]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {
    "ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
}
# Left at shipped defaults deliberately: target_false_positives_per_hour (0.2),
# max_negative_weight (1500), batch_n_per_class, model_type/layer_size.
# (target_accuracy / target_recall from the upstream notebook are NOT read by train.py.)

yaml.dump(config, open("edmund.yaml", "w"))
print(yaml.dump(config, sort_keys=True))

## 4. Train

Three stages. Each is resumable — generation counts what already exists and tops it up.

In [ ]:
import re, pathlib

# PyTorch 2.6 flipped torch.load's `weights_only` default to True. The LibriTTS
# checkpoint is a pickled SynthesizerTrn *module*, not a bare state_dict, so the
# safe unpickler cannot load it at all -- allowlisting the class would work too,
# but the flag states the same trust decision once.
#
# The trust question is real and the answer here is yes: this is the official
# rhasspy v1.0.0 release asset, fetched over HTTPS in the environment cell.
p = pathlib.Path("piper-sample-generator/generate_samples.py")
s = p.read_text()
if "weights_only" in s:
    print("already patched")
else:
    s2 = re.sub(r"torch\.load\(\s*model_path\s*\)",
                "torch.load(model_path, weights_only=False)", s)
    assert s2 != s, "torch.load(model_path) call not found -- upstream changed"
    p.write_text(s2)
    print("patched:", [l.strip() for l in s2.split("\n") if "torch.load" in l])

In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py --training_config edmund.yaml --generate_clips

In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py --training_config edmund.yaml --augment_clips

In [ ]:
# Only the small classification head trains; the feature extractor stays frozen.
!{sys.executable} openwakeword/openwakeword/train.py --training_config edmund.yaml --train_model

## 5. Collect the model

In [ ]:
import glob
found = glob.glob("./my_custom_model/edmund.onnx") + glob.glob("**/edmund.onnx", recursive=True)
found = sorted(set(found))
for p in found:
    print(p, os.path.getsize(p), "bytes")
if found:
    from google.colab import files
    files.download(found[0])
else:
    print("No edmund.onnx found — check the training output above.")

Send `edmund.onnx` back to Claude.

What happens next: the ONNX goes to the Pi; `mirror-wake.py` drops Vosk and the RMS level
gate for openWakeWord with Silero VAD gating; the threshold is tuned against real room audio
toward <0.5 false accepts/hour; and the edit-distance check in `src/mirror/utterance.ts` gets
deleted, because verification becomes a number instead of a list of spellings.